# Training a small MLP controller

`1104_loss_landscape_minima.ipynb` optimized the parameters of a single reference directly. There
the output head has nothing to model: the best answer is one point. This notebook trains a
network instead, so one set of weights has to serve many references. Does the categorical
Gumbel head help over a plain regression head?

Setup: random references over the full trainable ranges, each a short constant clip. The input is
the reference's middle-frame MFCCs plus its pitch (pitch is given, as in training, where it comes
from pyin). A small MLP predicts the 7 trainable parameters, the synth renders them, and the loss
is the middle-frame MFCC distance to the reference, as in the earlier notebooks.

Three heads, same MLP body and the same training budget:
- **direct**: one scalar per parameter squashed into its range by a scaled tanh, like
  `head_type="direct"` in the real model.
- **gumbel soft**: the training setup. Logits over 32 bucket centers, soft Gumbel-softmax at
  tau = 2, the synth hears the weighted mean of the centers, plus the hinged entropy penalty.
- **gumbel hard**: the same, but straight-through (`gumbel_hard=True`). The synth hears one
  sampled bucket center, so a head that keeps mass on two different solutions is scored on each
  of them rather than on their blend.

All heads start at mid-range (zero output bias), as in the real model. The Gumbel heads are
evaluated both at the argmax bucket (what eval does) and at the expected value.

By default the references and the training synth share the noise seed, so every reference can be
matched exactly. Set `DIFFERENT_NOISE = True` to give each reference its own seed: then, like a
real recording, part of the target can't be predicted from the input.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import plotly.graph_objects as go
import torch
import torch.nn.functional as F
from IPython.display import Audio, Markdown, display
from torch import nn

from samuel.losses import MFCCLoss
from samuel.model import _DEFAULT_FROZEN_VALUES, _DEFAULT_PARAM_SPEC, PinkTromboneControllerConfig
from samuel.pink_trombone import PARAM_NAMES, SAMPLE_RATE, pink_trombone_ola

SAMPLES_PER_FRAME = 512
CONTROL_RATE = SAMPLE_RATE / SAMPLES_PER_FRAME
# The loss clip is 4096 samples -> 5 MFCC frames; the middle one covers samples 1024-3072.
# Parameters are constant, so the clip is synthesized as a single 4096-sample control frame.
LOSS_CLIP_SAMPLES = 4096
SEED = 0
DIFFERENT_NOISE = False
DEVICE = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

N_TRAIN = 4096
N_TEST = 256
FREQUENCY_RANGE = (80.0, 220.0)

HIDDEN = 256
N_STEPS = 1500
BATCH_SIZE = 64
LR = 1e-3
EVAL_EVERY = 250

# Gumbel heads, matching configs/config.yaml.
N_BUCKETS = PinkTromboneControllerConfig().n_buckets  # 32
TAU = 2.0
ENTROPY_WEIGHT = 0.1
ENTROPY_FLOOR = 1.0  # nats

# The loss stays on CPU: torch.stft on MPS crashed the kernel. It's cheap next to the synth.
loss_fn = MFCCLoss(samples_per_frame=SAMPLES_PER_FRAME, n_fft=2048)

## References

In [ ]:
TRAINABLE = list(_DEFAULT_PARAM_SPEC)
lo = torch.tensor([_DEFAULT_PARAM_SPEC[n][0] for n in TRAINABLE])
hi = torch.tensor([_DEFAULT_PARAM_SPEC[n][1] for n in TRAINABLE])
TRAINABLE_IDX = [PARAM_NAMES.index(n) for n in TRAINABLE]
FREQ_IDX = PARAM_NAMES.index("frequency")
BASE = torch.tensor([_DEFAULT_FROZEN_VALUES.get(n, 0.0) for n in PARAM_NAMES])
assert set(PARAM_NAMES) == {*TRAINABLE, *_DEFAULT_FROZEN_VALUES, "frequency"}


def make_params(u: torch.Tensor, freq: torch.Tensor, n_frames: int = 1) -> torch.Tensor:
    """u [B, n_trainable] in [0, 1] and freq [B] -> [B, n_frames, 12], constant over time."""
    values = lo + u * (hi - lo)
    params = BASE.expand(len(u), -1).clone()
    params[:, TRAINABLE_IDX] = values
    params[:, FREQ_IDX] = freq
    return params[:, None].expand(-1, n_frames, -1).to(DEVICE)


def synth(
    params: torch.Tensor, seeds: torch.Tensor | None = None,
    control_rate: float = SAMPLE_RATE / LOSS_CLIP_SAMPLES,
) -> torch.Tensor:
    # A [B] seed tensor: equal seeds give every row the same noise.
    if seeds is None:
        seeds = torch.full((len(params),), SEED)
    return pink_trombone_ola(params, seed=seeds, control_rate=control_rate).cpu()


def middle_frame_features(audio: torch.Tensor) -> torch.Tensor:
    """[B, S] -> [B, n_mfcc] MFCCs of the middle frame."""
    feats = loss_fn.features(audio)  # [B, n_mfcc, T]
    return feats[..., feats.shape[-1] // 2]


def listen(u: torch.Tensor, freq: torch.Tensor) -> Audio:
    """1 s of audio for one row."""
    with torch.no_grad():
        audio = synth(make_params(u, freq, n_frames=round(CONTROL_RATE)), control_rate=CONTROL_RATE)
    return Audio(audio[0].numpy(), rate=SAMPLE_RATE)

In [ ]:
gen = torch.Generator().manual_seed(SEED)
n_refs = N_TRAIN + N_TEST
ref_u = torch.rand(n_refs, len(TRAINABLE), generator=gen)
ref_freq = FREQUENCY_RANGE[0] + (FREQUENCY_RANGE[1] - FREQUENCY_RANGE[0]) * torch.rand(n_refs, generator=gen)
ref_seeds = (
    torch.randint(1, 2**31 - 1, (n_refs,), generator=gen) if DIFFERENT_NOISE
    else torch.full((n_refs,), SEED)
)

with torch.no_grad():
    ref_feats = torch.cat([
        middle_frame_features(synth(make_params(u, f), s))
        for u, f, s in zip(ref_u.split(256), ref_freq.split(256), ref_seeds.split(256))
    ])

# Network input: standardized MFCCs and log pitch.
inputs = torch.cat([ref_feats, ref_freq.log()[:, None]], dim=1)
mean, std = inputs[:N_TRAIN].mean(0), inputs[:N_TRAIN].std(0)
inputs = (inputs - mean) / std
train_idx = torch.arange(N_TRAIN)
test_idx = torch.arange(N_TRAIN, n_refs)
print(f"{N_TRAIN} train, {N_TEST} test references, input dim {inputs.shape[1]}")

## Model

The MLP and the bucket centers live on the CPU (the network is tiny); only the synth runs on
`DEVICE`. Values are in normalized units, 0-1 across each parameter's range.

In [ ]:
centers = torch.linspace(0.0, 1.0, N_BUCKETS)


class Controller(nn.Module):
    def __init__(self, head: str):
        super().__init__()
        self.head = head
        n_out = len(TRAINABLE) * (1 if head == "direct" else N_BUCKETS)
        self.body = nn.Sequential(
            nn.Linear(inputs.shape[1], HIDDEN), nn.GELU(),
            nn.Linear(HIDDEN, HIDDEN), nn.GELU(),
            nn.Linear(HIDDEN, n_out),
        )
        with torch.no_grad():
            self.body[-1].bias.zero_()  # mid-range start, as in the real model

    def logits(self, x: torch.Tensor) -> torch.Tensor:
        return self.body(x).view(len(x), len(TRAINABLE), N_BUCKETS)

    def sample(self, x: torch.Tensor) -> torch.Tensor:
        """Training-time values [B, n_trainable]."""
        if self.head == "direct":
            return 0.5 + 0.5 * torch.tanh(self.body(x))
        weights = F.gumbel_softmax(self.logits(x), tau=TAU, hard=self.head == "gumbel hard", dim=-1)
        return (weights * centers).sum(-1)

    @torch.no_grad()
    def predict(self, x: torch.Tensor) -> dict[str, torch.Tensor]:
        """Eval-time values, by evaluation mode."""
        if self.head == "direct":
            return {"value": 0.5 + 0.5 * torch.tanh(self.body(x))}
        logits = self.logits(x)
        return {
            "argmax": centers[logits.argmax(-1)],
            "expected": (logits.softmax(-1) * centers).sum(-1),
        }


def ref_loss(u: torch.Tensor, idx: torch.Tensor) -> torch.Tensor:
    """[B] middle-frame MFCC loss of values u against references idx (training noise seed)."""
    feats = middle_frame_features(synth(make_params(u, ref_freq[idx])))
    return (feats - ref_feats[idx]).abs().mean(dim=-1)


@torch.no_grad()
def evaluate(model: Controller, idx: torch.Tensor) -> dict[str, torch.Tensor]:
    """Per-reference loss for each evaluation mode."""
    preds = model.predict(inputs[idx])
    return {
        mode: torch.cat([ref_loss(u_chunk, i_chunk) for u_chunk, i_chunk in zip(u.split(128), idx.split(128))])
        for mode, u in preds.items()
    }

## Training

Each head is trained and evaluated in turn, so results appear as they come in. The training loss
of the Gumbel heads is measured at the sampled values.

In [ ]:
HEADS = ["direct", "gumbel soft", "gumbel hard"]
models, histories, test_curves = {}, {}, {}


def train(head: str) -> Controller:
    torch.manual_seed(SEED)
    model = Controller(head)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    batch_gen = torch.Generator().manual_seed(SEED)
    history, test_curve = [], []
    for step in range(N_STEPS + 1):
        if step % EVAL_EVERY == 0:
            model.eval()
            test = {mode: losses.median().item() for mode, losses in evaluate(model, test_idx).items()}
            test_curve.append((step, test))
            print(f"step {step:4d}: test median " + ", ".join(f"{m} {v:.3f}" for m, v in test.items()))
            model.train()
        if step == N_STEPS:
            break
        idx = train_idx[torch.randint(N_TRAIN, (BATCH_SIZE,), generator=batch_gen)]
        loss = ref_loss(model.sample(inputs[idx]), idx).mean()
        if head != "direct":
            log_probs = model.logits(inputs[idx]).log_softmax(-1)
            entropy = -(log_probs.exp() * log_probs).sum(-1)
            loss = loss + ENTROPY_WEIGHT * F.relu(ENTROPY_FLOOR - entropy).mean()
        history.append(loss.item())
        opt.zero_grad()
        loss.backward()
        opt.step()
    histories[head], test_curves[head] = history, test_curve
    return model.eval()

In [ ]:
print("direct")
models["direct"] = train("direct")

In [ ]:
print("gumbel soft")
models["gumbel soft"] = train("gumbel soft")

In [ ]:
print("gumbel hard")
models["gumbel hard"] = train("gumbel hard")

## Results

In [ ]:
THRESHOLD = 0.05
untrained = evaluate(Controller("direct").eval(), test_idx)["value"]  # mid-range everywhere
final = {"untrained (mid-range)": untrained}
for head, model in models.items():
    for mode, losses in evaluate(model, test_idx).items():
        final[head if mode == "value" else f"{head} ({mode})"] = losses

rows = [
    f"| {name} | {losses.median():.3f} | {losses.mean():.3f} | {(losses < THRESHOLD).float().mean():.0%} |"
    for name, losses in final.items()
]
display(Markdown(
    f"Test set, {N_TEST} references.\n\n"
    f"| head | median loss | mean loss | references below {THRESHOLD} |\n"
    "|---|---|---|---|\n" + "\n".join(rows)
))

Results of the MPS run on 2026-11-05 (`DIFFERENT_NOISE = False`):

| head | median loss | mean loss | references below 0.05 |
|---|---|---|---|
| untrained (mid-range) | 0.196 | 0.225 | 2% |
| direct | 0.007 | 0.024 | 83% |
| gumbel soft (argmax) | 0.009 | 0.044 | 72% |
| gumbel soft (expected) | 0.006 | 0.030 | 81% |
| gumbel hard (argmax) | 0.012 | 0.107 | 59% |
| gumbel hard (expected) | 0.012 | 0.096 | 59% |

In [ ]:
fig = go.Figure()
for name, losses in final.items():
    fig.add_trace(go.Histogram(x=losses, name=name, nbinsx=40, opacity=0.5))
fig.update_layout(
    barmode="overlay", height=420, xaxis_title="MFCC loss (middle frame)", yaxis_title="test references",
    title="Test loss per reference",
)
fig.show()

In [ ]:
fig = go.Figure()
window = 50
for head, history in histories.items():
    smoothed = torch.tensor(history).unfold(0, window, 1).mean(-1)
    fig.add_trace(go.Scatter(x=torch.arange(window - 1, len(history)), y=smoothed, name=f"{head} (train)"))
for head, curve in test_curves.items():
    for mode in curve[0][1]:
        fig.add_trace(go.Scatter(
            x=[s for s, _ in curve], y=[t[mode] for _, t in curve], mode="lines+markers",
            line_dash="dot", name=f"{head} (test {mode} median)",
        ))
fig.update_layout(height=420, xaxis_title="step", yaxis_title="loss", title="Training curves")
fig.show()

### Does the Gumbel head use its uncertainty?

Per test reference and parameter: the softmax entropy, and how often the distribution is
bimodal, i.e. its top two buckets each hold at least 20% of the mass and are at least 4 buckets
apart. If the categorical head is modelling several solutions, the bimodal fraction should be well
above zero, and the argmax and expected value should then sound different.

In [ ]:
rows = []
for head in ["gumbel soft", "gumbel hard"]:
    with torch.no_grad():
        probs = models[head].logits(inputs[test_idx]).softmax(-1)  # [N_TEST, n_trainable, N_BUCKETS]
    entropy = -(probs * probs.clamp_min(1e-12).log()).sum(-1)
    top = probs.topk(2, dim=-1)
    bimodal = (top.values[..., 1] >= 0.2) & ((top.indices[..., 0] - top.indices[..., 1]).abs() >= 4)
    rows.append(
        f"| {head} | {entropy.mean():.2f} | {bimodal.float().mean():.1%} | "
        + " | ".join(f"{bimodal[:, j].float().mean():.0%}" for j in range(len(TRAINABLE))) + " |"
    )
display(Markdown(
    "| head | mean entropy (nats) | bimodal | " + " | ".join(TRAINABLE) + " |\n"
    + "|---" * (3 + len(TRAINABLE)) + "|\n" + "\n".join(rows)
))

### Listening

A few test references, each followed by every head's eval output (argmax for the Gumbel
heads).

In [ ]:
for i in test_idx[:5].tolist():
    ref_values = lo + ref_u[i] * (hi - lo)
    display(Markdown(
        f"**Reference {i}** (frequency {ref_freq[i]:.0f} Hz): "
        + ", ".join(f"{n}={v:.2f}" for n, v in zip(TRAINABLE, ref_values.tolist()))
    ))
    display(listen(ref_u[i : i + 1], ref_freq[i : i + 1]))
    for head, model in models.items():
        mode, u = next(iter(model.predict(inputs[i : i + 1]).items()))
        loss = final[head if mode == "value" else f"{head} ({mode})"][i - N_TRAIN]
        values = lo + u[0] * (hi - lo)
        display(Markdown(
            f"*{head}* (loss {loss:.3f}): "
            + ", ".join(f"{n}={v:.2f}" for n, v in zip(TRAINABLE, values.tolist()))
        ))
        display(listen(u, ref_freq[i : i + 1]))